# Phase 1 — Lesson 04: Calculus for Machine Learning
> Faithful study notes built directly from the lesson source. Concepts, code, diagrams, and exercises.

**Source:** ai-engineering-from-scratch -> Phase 1 (Math Foundations), Lesson 04: Calculus for Machine Learning
**Type:** Learn | **Languages:** Python | **Time:** ~60 minutes
**Prerequisites:** Phase 1, Lessons 01-03



# What You'll Learn

---

## Learning Objectives

- Compute numerical and analytical derivatives for common ML functions (x², sigmoid, cross-entropy)
- Implement gradient descent from scratch to minimize a loss function in 1D and 2D
- Derive the gradient of a linear regression model and train it via manual weight updates


# The Problem

---

You have a neural network with millions of weights. Each weight is a knob. You need to figure out which direction to turn every single knob to make the model slightly less wrong. Calculus gives you that direction.


# The Derivative Measures Rate of Change

---

A derivative measures the rate of change. For a function y = f(x), the derivative f'(x) tells you: if you nudge x by a tiny amount, how much does y change?

Geometrically, the derivative is the slope of the tangent line at a point.


In [ ]:
import numpy as npimport matplotlib.pyplot as pltdef numerical_derivative(f, x, h=1e-5):    return (f(x + h) - f(x - h)) / (2 * h)x = np.linspace(-3, 3, 400)y = x ** 2dy = np.array([numerical_derivative(lambda v: v**2, xi) for xi in x])fig, ax = plt.subplots(figsize=(7, 4))ax.plot(x, y, label='f(x) = x²', color='#2c3e50', lw=2)ax.plot(x, dy, label="f'(x) ≈ 2x", color='#e74c3c', lw=2, linestyle='--')ax.axhline(0, color='gray', lw=0.8); ax.axvline(0, color='gray', lw=0.8)ax.set_xlabel('x'); ax.set_ylabel('value')ax.set_title('Function and Its Derivative', fontsize=12, fontweight='bold')ax.legend(); ax.grid(True, alpha=0.3)plt.tight_layout(); plt.show()# Show derivative at specific pointsfor xv in [-2, -1, 0, 1, 2]:    print(f"x={xv:2d}  f(x)={xv**2:3d}  f'(x) ≈ {numerical_derivative(lambda v: v**2, xv):.1f}")

# The Gradient: Vector of All Partial Derivatives

---

Real functions have many inputs. A neural network loss depends on thousands of weights. A partial derivative holds all variables constant except one, then takes the derivative with respect to that one.

The gradient collects every partial derivative into one vector. The gradient points in the direction of steepest ascent. To minimize a function, go in the opposite direction.


In [ ]:
import numpy as npimport matplotlib.pyplot as pltdef f_2d(point):    x, y = point    return x**2 + y**2def numerical_gradient(f, point, h=1e-5):    grad = []    for i in range(len(point)):        p_plus = list(point); p_minus = list(point)        p_plus[i] += h; p_minus[i] -= h        grad.append((f(p_plus) - f(p_minus)) / (2 * h))    return grad# Gradient descent on 2D bowlpoint = [4.0, 3.0]lr = 0.1path = [point[:]]for _ in range(30):    grad = numerical_gradient(f_2d, point)    point = [p - lr * g for p, g in zip(point, grad)]    path.append(point[:])path = np.array(path)# Contour plotx = np.linspace(-5, 5, 50)y = np.linspace(-5, 5, 50)X, Y = np.meshgrid(x, y)Z = X**2 + Y**2fig, ax = plt.subplots(figsize=(6, 6))cont = ax.contour(X, Y, Z, levels=20, cmap='viridis', alpha=0.6)ax.plot(path[:, 0], path[:, 1], 'ro-', markersize=4, label='Gradient descent path')ax.plot(path[0, 0], path[0, 1], 'ro', markersize=8, label='Start')ax.plot(path[-1, 0], path[-1, 1], 'go', markersize=8, label='End (near 0,0)')ax.set_xlabel('x'); ax.set_ylabel('y')ax.set_title('Gradient Descent on f(x,y) = x² + y²', fontsize=12, fontweight='bold')ax.legend(); ax.grid(True, alpha=0.3)plt.tight_layout(); plt.show()print(f"Start: ({path[0,0]:.2f}, {path[0,1]:.2f})  f={f_2d(path[0]):.4f}")print(f"End:   ({path[-1,0]:.4f}, {path[-1,1]:.4f})  f={f_2d(path[-1]):.6f}")

In [ ]:
import mathdef sigmoid(x): return 1 / (1 + math.exp(-x))def sigmoid_derivative(x):    s = sigmoid(x)    return s * (1 - s)for xv in [-3, 0, 3]:    print(f"sigmoid({xv}) = {sigmoid(xv):.4f}, sigmoid'({xv}) = {sigmoid_derivative(xv):.4f}")

# Gradient Descent Trains a Linear Model

---

Training a linear regression model y = wx + b by gradient descent:

- Predict: `pred = w*x + b`
- Compute loss: `loss = mean((pred - y)²)`
- Compute gradients: `dw = 2*mean(error*x)`, `db = 2*mean(error)`
- Update: `w -= lr*dw`, `b -= lr*db`


In [ ]:
import numpy as npimport matplotlib.pyplot as pltnp.random.seed(42)xs = np.array([1.0, 2.0, 3.0, 4.0, 5.0])ys = np.array([3.0, 5.0, 7.0, 9.0, 11.0])w, b = np.random.randn(), np.random.randn()lr = 0.01losses = []ws, bs = [], []for epoch in range(200):    pred = w * xs + b    error = pred - ys    loss = np.mean(error ** 2)    dw = np.mean(2 * error * xs)    db = np.mean(2 * error)    w -= lr * dw    b -= lr * db    losses.append(loss)    ws.append(w); bs.append(b)plt.figure(figsize=(7, 4))plt.plot(losses, color='#e74c3c', lw=2)plt.xlabel('Epoch'); plt.ylabel('Loss (MSE)')plt.title('Gradient Descent Trains Linear Model', fontsize=12, fontweight='bold')plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()print(f"Learned: y = {w:.2f}x + {b:.2f}")print(f"Actual:  y = 2x + 1")

# Quiz Review

---

Try answering these before reading the explanations:

**Q1:** What does the derivative tell you in ML?
> **Explanation:** Which way is downhill — the negative gradient reduces the loss.

**Q2:** What is the gradient?
> **Explanation:** A vector of all partial derivatives — points uphill; negative gradient = downhill.

**Q3:** What does Bessel's correction do?
> **Explanation:** Divides sample variance by (n-1) instead of n to make it an unbiased estimator.



# Practice Exercises

---

Do these in order:

**Exercise 1:** Use gradient descent to find the minimum of f(x, y) = (x - 3)² + (y + 1)² starting from (0, 0).

**Exercise 2:** Compare convergence with and without momentum on f(x) = x⁴ - 3x².



# Key Terms

---

| Term | What people say | What it actually means |
|------|----------------|----------------------|
| Derivative | The slope | Rate of change of a function at a point |
| Gradient | Direction of steepest ascent | Vector of all partial derivatives |
| Gradient descent | Go downhill | Subtract gradient × learning rate to reduce loss |
| Learning rate | Step size | Controls how big each gradient descent step is |
| Backpropagation | Reverse-mode autodiff | Computing gradients layer by layer from output to input |


# Further Reading

---

- [3Blue1Brown: Essence of Calculus](https://www.3blue1brown.com/topics/calculus)
- [Stanford CS231n: Backpropagation](https://cs231n.github.io/optimization-2/)


# Next Steps

---

When you're ready, continue to Lesson 05: Chain Rule & Automatic Differentiation.

Your `ai-engineering-from-scratch` learning directory is **completely untouched** — these notes live separately in this folder.
